# CUDA MacroV2 TinyStories Run

This notebook runs the MacroV2 lane on **TinyStories**, not synthetic data.

It clones/pulls the repo into the Colab runtime, installs the data dependencies, verifies TinyStories streaming works, builds TinyStories configs, trains or reuses a recursive exact teacher checkpoint, then trains MacroV2 against that teacher.

Defaults are deliberately moderate. Change the env-style values in the config cell before running if you want a longer run.


In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path


def _run(cmd, *, cwd=None, env=None):
    print("$", " ".join(map(str, cmd)))
    subprocess.check_call(list(map(str, cmd)), cwd=str(cwd) if cwd else None, env=env)

REPO_URL = os.environ.get("REPO_URL", "https://github.com/BrownHujay/Speedup-Thingy.git")
REPO_BRANCH = os.environ.get("REPO_BRANCH", "master")
REPO_ROOT = Path(os.environ.get("REPO_ROOT", "/content/Speedup-Thingy"))

if not REPO_ROOT.exists():
    _run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_ROOT])
else:
    _run(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_ROOT)
    _run(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT)
    _run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=REPO_ROOT)

sys.path.insert(0, str(REPO_ROOT / "src"))
print("repo_root:", REPO_ROOT)


In [ ]:
# Dependencies needed for TinyStories + GPT2 BPE tokenization.
_run([sys.executable, "-m", "pip", "install", "-q", "datasets", "tokenizers", "pyyaml", "pandas"])

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))


In [ ]:
# Safety patch for cloned repos that predate strict dataset loading.
# Without this, the loader can silently fall back to built-in toy text if HF/datasets fails.
data_py = REPO_ROOT / "src" / "recursive_training_engine" / "data.py"
text = data_py.read_text()
if "RTE_STRICT_DATASET" not in text:
    text = text.replace("from itertools import islice
", "from itertools import islice
import os
")
    text = text.replace(
        "    except Exception:
        return [
",
        "    except Exception:
"
        "        if os.environ.get("RTE_STRICT_DATASET", "").strip().lower() in {
"
        "            "1",
"
        "            "true",
"
        "            "yes",
"
        "            "y",
"
        "            "on",
"
        "        }:
"
        "            raise
"
        "        return [
",
    )
    data_py.write_text(text)
    print("patched strict TinyStories fallback guard into", data_py)
else:
    print("strict dataset guard already present")

os.environ["RTE_STRICT_DATASET"] = "1"


In [ ]:
from datasets import load_dataset

# Hard fail here if TinyStories is not reachable. This keeps us honest.
ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
first = next(iter(ds))["text"]
print("TinyStories sample chars:", len(first))
print(first[:500].replace("
", " "))


## Run Configuration

The defaults are the existing MacroV2 proof-ish size (`d=64`, `d_ff=256`, depth 4) on TinyStories with GPT2 BPE filtered to vocab 2048. Set `VOCAB_PROJECTION=modulo` before running if you want folded GPT2 vocab instead of filtered low-id tokens.


In [ ]:
def _bool_env(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "y", "on"}

D_MODEL = int(os.environ.get("D_MODEL", "64"))
D_FF = int(os.environ.get("D_FF", "256"))
N_LAYERS = int(os.environ.get("N_LAYERS", "11"))
N_HEADS = int(os.environ.get("N_HEADS", "8"))
VOCAB_SIZE = int(os.environ.get("VOCAB_SIZE", "2048"))
SEQ_LEN = int(os.environ.get("SEQ_LEN", "64"))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "128"))
PRECISION = os.environ.get("PRECISION", "fp32")
SEED = int(os.environ.get("SEED", "1234"))

TRAIN_TOKENS = int(os.environ.get("TRAIN_TOKENS", "6000000"))
EVAL_TOKENS = int(os.environ.get("EVAL_TOKENS", "65536"))
VOCAB_PROJECTION = os.environ.get("VOCAB_PROJECTION", "filter")  # filter or modulo

# Existing MacroV2 path: exact teacher first, then train the macro endpoint.
EXACT_TEACHER_STEPS = int(os.environ.get("EXACT_TEACHER_STEPS", "1000"))
MACROV2_STEPS = int(os.environ.get("MACROV2_STEPS", "1000"))
LOG_EVERY = int(os.environ.get("LOG_EVERY", "50"))
FIXED_RECIPE = int(os.environ.get("FIXED_RECIPE", "1"))
FIXED_DEPTH = int(os.environ.get("FIXED_DEPTH", "4"))
TEACHER_MODE = os.environ.get("TEACHER_MODE", "exact")  # exact or deferred_grouped

RUN_EXACT_TEACHER = _bool_env("RUN_EXACT_TEACHER", True)
RUN_MACROV2 = _bool_env("RUN_MACROV2", True)
RUN_EVALS = _bool_env("RUN_EVALS", True)
RESUME_EXISTING = _bool_env("RESUME_EXISTING", True)

# If you already have a teacher checkpoint on this Colab machine, set this path.
TEACHER_CHECKPOINT_OVERRIDE = os.environ.get("TEACHER_CHECKPOINT", "").strip()

RUN_STAMP = os.environ.get("RUN_STAMP", time.strftime("%Y%m%d_%H%M%S"))
RUN_ROOT = REPO_ROOT / "runs" / f"colab_macrov2_tinystories_d{D_MODEL}_h{D_FF}_{RUN_STAMP}"
CONFIG_DIR = RUN_ROOT / "configs"
CHECKPOINT_DIR = RUN_ROOT / "checkpoints"
REPORT_DIR = RUN_ROOT / "reports"
for p in (CONFIG_DIR, CHECKPOINT_DIR, REPORT_DIR):
    p.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "run_root": str(RUN_ROOT),
    "dataset": "tinystories",
    "projection": VOCAB_PROJECTION,
    "d_model": D_MODEL,
    "d_ff": D_FF,
    "layers": N_LAYERS,
    "heads": N_HEADS,
    "batch_size": BATCH_SIZE,
    "seq_len": SEQ_LEN,
    "train_tokens": TRAIN_TOKENS,
    "eval_tokens": EVAL_TOKENS,
    "exact_teacher_steps": EXACT_TEACHER_STEPS,
    "macrov2_steps": MACROV2_STEPS,
}, indent=2))


In [ ]:
import yaml


def _divisor_at_most(value: int, limit: int) -> int:
    candidate = min(value, limit)
    while candidate > 1 and value % candidate != 0:
        candidate -= 1
    return max(1, candidate)

ffn_groups = _divisor_at_most(D_FF, 16 if D_FF <= 256 else 64)
head_groups = _divisor_at_most(N_HEADS, N_HEADS)
active_head_groups = min(2, head_groups)
active_ffn_groups = min(2, ffn_groups)
router_hidden = max(64, D_MODEL // 2)
macro_rank = int(os.environ.get("MACRO_RANK", str(max(1, min(64, D_MODEL)))))
macro_hidden_mult = int(os.environ.get("MACRO_HIDDEN_MULT", "4"))

base_model = {
    "topology": "recursive",
    "vocab_size": VOCAB_SIZE,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "d_ff": D_FF,
    "n_dense_layers": N_LAYERS,
    "n_prelude": 1,
    "n_coda": 1,
    "t_max": FIXED_DEPTH,
    "depth_choices": sorted(set([1, 2, FIXED_DEPTH])),
    "attn_banks": 1,
    "ffn_banks": 2,
    "head_groups": head_groups,
    "ffn_groups": ffn_groups,
    "active_head_groups": active_head_groups,
    "active_ffn_groups": active_ffn_groups,
    "recipe_count": 16,
    "tie_embeddings": True,
    "use_rope": True,
    "use_recursive_input_skip": True,
    "fairness_tolerance": 0.01,
    "macro_type": "v2_delta_radius",
    "macro_rank": macro_rank,
    "macro_hidden_mult": macro_hidden_mult,
    "macro_radius_init_from_teacher": True,
    "macro_radius_clamp_mult_min": 0.25,
    "macro_radius_clamp_mult_max": 4.0,
    "macro_use_depth_embedding": True,
    "macro_use_recipe_embedding": True,
    "macro_use_delta_to_h0": True,
    "macro_decomposition": "direct",
    "router_hidden": router_hidden,
}

base_training = {
    "mode": "recursive_exact",
    "optimizer": "adamw",
    "lr": float(os.environ.get("LR", "0.001")),
    "lr_base": float(os.environ.get("LR_BASE", "0.001")),
    "lr_macro": float(os.environ.get("LR_MACRO", "0.003")),
    "lr_coda": float(os.environ.get("LR_CODA", "0.0003")),
    "lr_output": float(os.environ.get("LR_OUTPUT", "0.0003")),
    "lr_router": 0.0,
    "weight_decay": float(os.environ.get("WEIGHT_DECAY", "0.1")),
    "batch_size": BATCH_SIZE,
    "seq_len": SEQ_LEN,
    "grad_accum_steps": 1,
    "grad_clip_norm": 1.0,
    "seed": SEED,
    "precision": PRECISION,
    "audit_p_min": 0.0,
    "audit_p_max": 0.0,
    "audit_alpha": 0.0,
    "audit_beta": 0.0,
    "audit_gamma": 0.0,
    "audit_mode": "metric_only",
    "audit_gradient_correction": False,
    "lambda_hidden_mse": 1.0,
    "lambda_hidden_cosine": 0.5,
    "lambda_logit_kl": 0.25,
    "lambda_norm": 0.05,
    "lambda_cons": 0.001,
    "distill_temperature": 2.0,
    "fixed_recipe": FIXED_RECIPE,
    "fixed_recipe_schedule": [1, 3, 5, 7][:FIXED_DEPTH],
    "fixed_depth": FIXED_DEPTH,
    "disable_router_aux": True,
    "debug_force_full_output": True,
    "coverage_min": 0.0,
    "log_every": LOG_EVERY,
    "compile_model": False,
    "fused_optimizer": False,
    "foreach_optimizer": False,
    "allow_tf32": True,
    "strict_cuda": True,
    "require_triton": False,
    "require_flash_attention": False,
}

macro_training = dict(base_training)
macro_training.update({
    "mode": "recursive_macro_distill_only",
    "audit_mode": "distill_only",
    "lambda_hidden_mse": 0.0,
    "lambda_hidden_cosine": 0.0,
    "lambda_logit_kl": 0.25,
    "lambda_norm": 0.0,
    "lambda_delta_dir": 2.0,
    "lambda_delta_rms": 2.0,
    "lambda_endpoint_normed": 1.0,
    "lambda_endpoint_raw": 0.05,
    "macro_rms_trust_region": True,
    "lambda_macro_rms_trust": 2.0,
})

data_cfg = {
    "dataset": "tinystories",
    "tokenizer": "gpt2_bpe",
    "vocab_projection": VOCAB_PROJECTION,
    "projection_lane": VOCAB_PROJECTION,
    "train_tokens": TRAIN_TOKENS,
    "max_tokens": TRAIN_TOKENS,
    "eval_tokens": EVAL_TOKENS,
    "seed": SEED,
}

exact_config = {
    "run_name": "tinystories_exact_teacher",
    "output_dir": str(RUN_ROOT),
    "model": base_model,
    "training": base_training,
    "output": {"mode": "full"},
    "data": data_cfg,
}
macro_config = {
    "run_name": "tinystories_macrov2",
    "output_dir": str(RUN_ROOT),
    "model": base_model,
    "training": macro_training,
    "output": {"mode": "full"},
    "data": data_cfg,
}

EXACT_CONFIG = CONFIG_DIR / "tinystories_exact_teacher.yaml"
MACRO_CONFIG = CONFIG_DIR / "tinystories_macrov2.yaml"
EXACT_CONFIG.write_text(yaml.safe_dump(exact_config, sort_keys=False))
MACRO_CONFIG.write_text(yaml.safe_dump(macro_config, sort_keys=False))

TEACHER_CHECKPOINT = Path(TEACHER_CHECKPOINT_OVERRIDE) if TEACHER_CHECKPOINT_OVERRIDE else CHECKPOINT_DIR / "exact_teacher.pt"
MACRO_CHECKPOINT = CHECKPOINT_DIR / "macrov2.pt"

print("EXACT_CONFIG", EXACT_CONFIG)
print("MACRO_CONFIG", MACRO_CONFIG)
print("TEACHER_CHECKPOINT", TEACHER_CHECKPOINT)
print("MACRO_CHECKPOINT", MACRO_CHECKPOINT)
print(EXACT_CONFIG.read_text()[:1600])


In [ ]:
# Verify the RTE loader is actually producing TinyStories tokens, not synthetic/fallback text.
os.environ["RTE_STRICT_DATASET"] = "1"
from recursive_training_engine.config import load_config
from recursive_training_engine.data import load_token_streams

cfg = load_config(EXACT_CONFIG)
streams = load_token_streams(cfg.data, cfg.training, cfg.model.vocab_size)
print("tokenizer:", streams.tokenizer_name)
print("fingerprint:", streams.data_fingerprint)
print("train tokens:", int(streams.train.numel()))
print("eval tokens:", int(streams.eval.numel()))
assert streams.data_fingerprint.startswith("tinystories:"), streams.data_fingerprint
assert "synthetic" not in streams.tokenizer_name, streams.tokenizer_name


In [ ]:
CLI = [sys.executable, "-c", "from recursive_training_engine.cli import main; main()"]
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO_ROOT / "src") + os.pathsep + ENV.get("PYTHONPATH", "")
ENV["RTE_STRICT_DATASET"] = "1"

def run_rte(*args):
    _run(CLI + list(args), cwd=REPO_ROOT, env=ENV)


## Train or Reuse the Exact Teacher

This trains the recursive exact teacher on TinyStories. If `TEACHER_CHECKPOINT` exists and `RESUME_EXISTING=True`, it skips retraining.


In [ ]:
if RUN_EXACT_TEACHER:
    if RESUME_EXISTING and TEACHER_CHECKPOINT.exists():
        print("teacher checkpoint exists; skipping exact teacher train:", TEACHER_CHECKPOINT)
    else:
        run_rte(
            "train",
            "--config", str(EXACT_CONFIG),
            "--mode", "recursive_exact",
            "--steps", str(EXACT_TEACHER_STEPS),
            "--save-checkpoint", str(TEACHER_CHECKPOINT),
            "--run-dir", str(RUN_ROOT / "exact_teacher_train"),
        )
else:
    print("RUN_EXACT_TEACHER=False")

assert TEACHER_CHECKPOINT.exists(), f"missing teacher checkpoint: {TEACHER_CHECKPOINT}"


In [ ]:
if RUN_EVALS:
    print("Exact teacher eval on TinyStories eval stream")
    run_rte(
        "evaluate",
        "--config", str(EXACT_CONFIG),
        "--mode", "recursive_exact",
        "--checkpoint", str(TEACHER_CHECKPOINT),
    )


## Train MacroV2 Against the TinyStories Teacher

This trains only the MacroV2 endpoint using the exact teacher checkpoint. It does not silently use synthetic data because `RTE_STRICT_DATASET=1` is inherited by the subprocess.


In [ ]:
if RUN_MACROV2:
    if RESUME_EXISTING and MACRO_CHECKPOINT.exists():
        print("macro checkpoint exists; skipping MacroV2 train:", MACRO_CHECKPOINT)
    else:
        run_rte(
            "train-macro-teacher",
            "--config", str(MACRO_CONFIG),
            "--teacher-checkpoint", str(TEACHER_CHECKPOINT),
            "--teacher-mode", TEACHER_MODE,
            "--steps", str(MACROV2_STEPS),
            "--fixed-recipe", str(FIXED_RECIPE),
            "--fixed-depth", str(FIXED_DEPTH),
            "--log-every", str(LOG_EVERY),
            "--save-checkpoint", str(MACRO_CHECKPOINT),
            "--run-dir", str(RUN_ROOT / "macrov2_train"),
        )
else:
    print("RUN_MACROV2=False")


In [ ]:
if RUN_EVALS and MACRO_CHECKPOINT.exists():
    print("MacroV2 eval on TinyStories eval stream")
    run_rte(
        "evaluate",
        "--config", str(MACRO_CONFIG),
        "--mode", "recursive_macro_distill_only",
        "--checkpoint", str(MACRO_CHECKPOINT),
    )
else:
    print("Macro eval skipped; checkpoint missing or RUN_EVALS=False")


## Results Scoreboard

These cells parse the actual run artifacts and show the important numbers directly. You should not have to dig through the training log.


In [ ]:
import pandas as pd
from IPython.display import display


def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    for line in path.read_text().splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return rows


def compact_columns(rows: list[dict], preferred: list[str]) -> pd.DataFrame:
    if not rows:
        return pd.DataFrame()
    cols = [c for c in preferred if any(c in r for r in rows)]
    if not cols:
        cols = sorted(rows[-1].keys())[:20]
    return pd.DataFrame([{c: r.get(c) for c in cols} for r in rows])

exact_metrics_path = RUN_ROOT / "exact_teacher_train" / "metrics.jsonl"
macro_metrics_path = RUN_ROOT / "macrov2_train" / "metrics.jsonl"
exact_rows = load_jsonl(exact_metrics_path)
macro_rows = load_jsonl(macro_metrics_path)

print("exact metrics:", exact_metrics_path, "rows", len(exact_rows))
print("macro metrics:", macro_metrics_path, "rows", len(macro_rows))

if exact_rows:
    exact_df = compact_columns(
        exact_rows,
        [
            "step", "mode", "loss", "nll_per_token", "exact_nll", "macro_nll",
            "tokens_seen", "lr", "grad_norm", "train_tokens_per_second",
        ],
    )
    print("Exact teacher first/last rows")
    display(pd.concat([exact_df.head(3), exact_df.tail(3)]).drop_duplicates())
else:
    print("No exact teacher metrics found yet.")

if macro_rows:
    macro_df = compact_columns(
        macro_rows,
        [
            "step", "loss", "exact_eval_nll", "hot_eval_nll", "hidden_cosine",
            "logit_kl", "endpoint_normed_cosine", "delta_cosine", "macro_grad_norm",
            "macro_delta_dir_loss", "macro_delta_rms_loss", "macro_endpoint_normed_loss",
            "macro_endpoint_raw_loss", "macro_rms_trust_loss",
        ],
    )
    if "exact_eval_nll" in macro_df.columns and "hot_eval_nll" in macro_df.columns:
        macro_df["macro_gap"] = macro_df["hot_eval_nll"] - macro_df["exact_eval_nll"]
    print("MacroV2 first/last rows")
    display(pd.concat([macro_df.head(3), macro_df.tail(5)]).drop_duplicates())
else:
    print("No MacroV2 metrics found yet.")


In [ ]:
# Captured evals: this turns the raw CLI JSON into a small table.
import re


def run_rte_capture(*args) -> str:
    cmd = CLI + list(args)
    print("$", " ".join(map(str, cmd)))
    proc = subprocess.run(
        list(map(str, cmd)),
        cwd=str(REPO_ROOT),
        env=ENV,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    if proc.returncode != 0:
        print(proc.stdout[-6000:])
        raise subprocess.CalledProcessError(proc.returncode, cmd, output=proc.stdout)
    return proc.stdout


def extract_json_objects(text: str) -> list[dict]:
    decoder = json.JSONDecoder()
    objs = []
    for match in re.finditer(r"[\[{]", text):
        idx = match.start()
        try:
            obj, end = decoder.raw_decode(text[idx:])
        except Exception:
            continue
        if isinstance(obj, dict):
            objs.append(obj)
    return objs


def eval_checkpoint(label: str, config_path: Path, mode: str, checkpoint: Path) -> dict:
    if not checkpoint.exists():
        return {"label": label, "exists": False, "checkpoint": str(checkpoint)}
    text = run_rte_capture(
        "evaluate",
        "--config", str(config_path),
        "--mode", mode,
        "--checkpoint", str(checkpoint),
    )
    objs = extract_json_objects(text)
    result = objs[-1] if objs else {}
    loss_per_sample = result.get("eval_loss_per_sample")
    row = {
        "label": label,
        "exists": True,
        "mode": result.get("mode", mode),
        "eval_loss_per_sample": loss_per_sample,
        "eval_nll_per_token": (loss_per_sample / SEQ_LEN) if loss_per_sample is not None else None,
        "checkpoint": str(checkpoint),
    }
    return row

score_rows = []
if RUN_EVALS:
    score_rows.append(eval_checkpoint("exact_teacher", EXACT_CONFIG, "recursive_exact", TEACHER_CHECKPOINT))
    score_rows.append(eval_checkpoint("macrov2", MACRO_CONFIG, "recursive_macro_distill_only", MACRO_CHECKPOINT))

score_df = pd.DataFrame(score_rows)
if not score_df.empty:
    if score_df["eval_nll_per_token"].notna().any():
        exact_nll = score_df.loc[score_df["label"] == "exact_teacher", "eval_nll_per_token"]
        if not exact_nll.empty and pd.notna(exact_nll.iloc[0]):
            score_df["gap_vs_exact_teacher"] = score_df["eval_nll_per_token"] - float(exact_nll.iloc[0])
    display(score_df)
else:
    print("RUN_EVALS=False, captured eval skipped.")


In [ ]:
# One-line final summary from training logs + captured evals.
summary = {
    "dataset": "tinystories",
    "strict_dataset": os.environ.get("RTE_STRICT_DATASET"),
    "run_root": str(RUN_ROOT),
    "teacher_checkpoint_exists": TEACHER_CHECKPOINT.exists(),
    "macro_checkpoint_exists": MACRO_CHECKPOINT.exists(),
}
if exact_rows:
    summary["exact_last"] = exact_rows[-1]
if macro_rows:
    last_macro = dict(macro_rows[-1])
    if "exact_eval_nll" in last_macro and "hot_eval_nll" in last_macro:
        last_macro["macro_gap"] = last_macro["hot_eval_nll"] - last_macro["exact_eval_nll"]
    summary["macro_last"] = last_macro
print(json.dumps(summary, indent=2, sort_keys=True, default=str))


In [ ]:
print(json.dumps({
    "run_root": str(RUN_ROOT),
    "exact_config": str(EXACT_CONFIG),
    "macro_config": str(MACRO_CONFIG),
    "teacher_checkpoint": str(TEACHER_CHECKPOINT),
    "macro_checkpoint": str(MACRO_CHECKPOINT),
    "dataset": "tinystories",
    "strict_dataset": os.environ.get("RTE_STRICT_DATASET"),
}, indent=2))
